In [1]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [ ]:
# 1. import database/data building connection to connect python with database that is sqlite3
conn = sqlite3.connect('customer_churn.db')
sql_query = ''' 
SELECT  name
FROM    sqlite_master
WHERE   type = 'table'
'''

tables = pd.read_sql(sql_query, conn)

# create data frame for each table:
for table_name in tables['name']:
    df = pd.read_sql(f'SELECT * FROM {table_name}', conn)
    globals()[f'df_{table_name}'] = df
    print(f'created dataframe: df_{table_name}')
conn.close()

created dataframe: df_db_customer
created dataframe: df_db_subscription
created dataframe: df_db_support


In [84]:
# print table names and column names
conn = sqlite3.connect('customer_churn.db')
for table_name in tables['name']:
    print(f'\nTable Name: {table_name}')
    # get column information
    column_query = f'PRAGMA table_info({table_name});'
    columns = pd.read_sql(column_query, conn)
    print('columns:')
    print(columns['name'].tolist())
# close connection
conn.close()


Table Name: db_customer
columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: db_subscription
columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: db_support
columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


In [ ]:
# 2. data cleaning
df_db_customer.head()
# df_db_customer.tail()
# df_db_customer.info()

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,None,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,None,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [ ]:
# a. rename column - name into customer name
# b. drop columns - intrests and pin code
# c. change data type - dob
# d. data standerdization - gendre
# e. fix missing values - country

In [ ]:
# a. rename column - name into customer name
df_db_customer.rename(columns={'name' : 'customer_name'},inplace= True)

In [ ]:
# b. drop columns - intrests and pin code
# df_db_customer.drop(df_db_customer.columns[-2:], axis = 1)
df_db_customer.drop(columns=['interests', 'pincode'], inplace= True)

In [48]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customerid     21 non-null     object        
 1   customer_name  21 non-null     object        
 2   country        21 non-null     object        
 3   state          21 non-null     object        
 4   gender         21 non-null     object        
 5   dob            21 non-null     datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 1.1+ KB


In [34]:
# c. change data type - dob
df_db_customer['dob']=pd.to_datetime(df_db_customer['dob'])

In [38]:
# d. data standerdization - gender
# df_db_customer['gender'].unique()
df_db_customer['gender']=df_db_customer['gender'].replace({'Men':'Male', 'Women':'Female'})

In [ ]:
df_db_customer

In [ ]:
# e. fix missing values - country
df_db_customer[df_db_customer['country'].isna()]

In [46]:
# country and state - unique value pair
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [51]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,None,None,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,None,None,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,None,None,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [50]:
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     object 
 1   subscription_start_date  21 non-null     object 
 2   subscription_type        21 non-null     object 
 3   renewal_date             21 non-null     object 
 4   plan_type                21 non-null     object 
 5   contract_type            21 non-null     object 
 6   cancellation_date        6 non-null      object 
 7   cancellation_reason      6 non-null      object 
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 1.9+ KB


In [55]:
date_col = ['subscription_start_date', 'renewal_date', 'cancellation_date']
df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     object        
 1   subscription_start_date  21 non-null     datetime64[ns]
 2   subscription_type        21 non-null     object        
 3   renewal_date             21 non-null     datetime64[ns]
 4   plan_type                21 non-null     object        
 5   contract_type            21 non-null     object        
 6   cancellation_date        6 non-null      datetime64[ns]
 7   cancellation_reason      6 non-null      object        
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
dtypes: datetime64[ns](3), float64(1), int64(2), object(5)
memory usage: 1.9+ KB


In [63]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28,N,60
1,0003-MKNFE,2024-08-28,Y,10
2,0013-EXCHZ,2024-01-20,Y,20
3,0013-MHZWF,2025-03-18,N,90
4,0013-SMEOE,2024-11-01,N,30


In [61]:
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      object        
 1   complaint_date  9 non-null      datetime64[ns]
 2   escalations     9 non-null      object        
 3   csat_score      9 non-null      int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 420.0+ bytes


In [58]:
df_db_support.drop(columns=['col_1', 'comment'],inplace=True)

In [60]:
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])

In [ ]:
# 3. feature engineering and data analysis:

In [ ]:
# create a new col using existing col - churn flag
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(),1,0)

In [64]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627,12,0
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91,1
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,None,6.99,210,34,0
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,None,22.99,1725,8,0
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88,1


In [ ]:
# first fix support table duplicate tables and merge
df = (df_db_subscription
            .merge(df_db_customer, on='customerid', how='left')
            .merge(df_db_support, on='customerid', how='left'))

In [87]:
df.tail()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag,customer_name,country,state,gender,dob,complaint_date,escalations,csat_score
18,0021-IKXGC,2021-07-07,Paid,2025-07-07,Standard,Annual,NaT,None,13.99,840,27,0,vishakha,India,Rajasthan,Female,2000-09-02,NaT,NaN,NaN
19,0022-TCJCI,2023-09-14,Refferal,2024-09-14,Basic,Monthly,2024-09-14,Forgot to cancel trial,16.99,42,99,1,raghvendra,India,Telangana,Male,1983-12-30,2024-09-13,Y,10.0
20,0022-TCJCI,2023-09-14,Refferal,2024-09-14,Basic,Monthly,2024-09-14,Forgot to cancel trial,16.99,42,99,1,raghvendra,India,Telangana,Male,1983-12-30,2024-09-14,N,90.0
21,0023-HGHWL,2020-06-23,Organic,2025-06-23,Premium,Annual,NaT,None,22.99,1955,7,0,rishabh,India,Uttar Pradesh,Male,1991-05-14,NaT,NaN,NaN
22,0023-UYUPN,2022-12-31,Paid,2025-12-31,Standard,Monthly,NaT,None,13.99,790,47,0,sudevi,India,Maharashtra,Female,1977-10-06,NaT,NaN,NaN


In [89]:
df_db_subscription['customerid'].nunique()

21

In [90]:
df_db_customer['customerid'].nunique()

21

In [91]:
df_db_support['customerid'].nunique()

7

In [101]:
df_db_support['customerid'].count

<bound method Series.count of 2    0013-EXCHZ
5    0017-IUDMW
1    0003-MKNFE
8    0022-TCJCI
6    0019-EFAEP
4    0013-SMEOE
3    0013-MHZWF
Name: customerid, dtype: object>

In [97]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [100]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid',keep = 'last' )

In [102]:
# merge
df = (df_db_subscription
            .merge(df_db_customer, on='customerid', how='left')
            .merge(df_db_support, on='customerid', how='left'))

In [104]:
df.shape

(21, 21)

In [ ]:
# data analysis

In [105]:
df.to_csv('exported_churn_data.csv', index=False)

In [ ]:
# 1. churn rate

In [106]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='object')

In [111]:
churn_rate = df['churn_flag'].mean()*100.
print('Churn Rate = ',round(churn_rate, 2),'%')

Churn Rate =  28.57 %
